# 06 — Validation & Research Report
Read all results, answer 9 research questions, generate final outputs.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
SEED = 42
np.random.seed(SEED)
for d in ['../outputs/figures','../outputs/tables','../outputs/kernels',
          '../outputs/logs','../data/processed','../data/raw']:
    os.makedirs(d, exist_ok=True)
print("Setup done.")

Setup done.


## Load All Results

In [2]:
from src.data import load_wine_quality, scale_features_to_pi, get_deterministic_sample, WINE_QUANTUM_FEATURES, WINE_TARGET_COL
from sklearn.preprocessing import StandardScaler
from src.classical import run_isolation_forest
from src.interestingness import quantum_anomaly_score, quantum_novelty, quantum_boundary, q_interestingness, weight_sensitivity_analysis, WEIGHT_CONFIGS
from src.metrics import anomaly_category_labels, centered_kernel_alignment, top_k_overlap, neighborhood_jaccard_summary

df = load_wine_quality('../data/raw/winequality-red.csv')
N = 150
idx, df_sample = get_deterministic_sample(df, N, seed=42)
K_q = np.load('../outputs/kernels/K_quantum.npy')
K_c = np.load('../outputs/kernels/K_classical.npy')
X_q4_std = StandardScaler().fit_transform(df_sample[WINE_QUANTUM_FEATURES].values)
X_q4_scaled = scale_features_to_pi(df_sample[WINE_QUANTUM_FEATURES].values)

QAS = quantum_anomaly_score(K_q)
CAS = run_isolation_forest(X_q4_std, random_state=42)
N_score = quantum_novelty(K_q, k=5)
B_score = quantum_boundary(K_q)
QI = q_interestingness(QAS, CAS, N_score, B_score)
labels = anomaly_category_labels(CAS, QAS)

anomaly_df   = pd.read_csv('../outputs/tables/anomaly_overlap.csv')
neighbor_df  = pd.read_csv('../outputs/tables/neighborhood_overlap.csv')
perm_df      = pd.read_csv('../outputs/tables/permutation_control.csv')
depth_df     = pd.read_csv('../outputs/tables/depth_robustness.csv')
fmap_df      = pd.read_csv('../outputs/tables/feature_map_comparison.csv')
scale_df     = pd.read_csv('../outputs/tables/scalability.csv')
cross_df     = pd.read_csv('../outputs/tables/cross_dataset_results.csv')
top20_df     = pd.read_csv('../outputs/tables/top_q_interesting_observations.csv')
print("All result files loaded.")

Loaded wine quality: 1599 rows × 12 columns


FileNotFoundError: [Errno 2] No such file or directory: '../outputs/tables/permutation_control.csv'

## Compute Key Metrics

In [ ]:
cka_val = centered_kernel_alignment(K_q, K_c)
jac_summary = neighborhood_jaccard_summary(X_q4_std, K_q, k_values=[5,10,15])
top5_ov  = top_k_overlap(CAS, QAS, 5)
top10_ov = top_k_overlap(CAS, QAS, 10)
top20_ov = top_k_overlap(CAS, QAS, 20)
n_quantum_only = int((labels=="Quantum-only").sum())

sens = weight_sensitivity_analysis(QAS, CAS, N_score, B_score, WEIGHT_CONFIGS)
rank_stability = float(np.mean(sens["spearman_matrix"][np.triu_indices(len(WEIGHT_CONFIGS),k=1)]))

perm_sub = perm_df[perm_df['seed'].astype(str).str.isdigit()]
perm_mean = perm_sub['permutation_cka'].astype(float).mean()
perm_std  = perm_sub['permutation_cka'].astype(float).std()
z_score   = (cka_val - perm_mean) / perm_std if perm_std > 0 else float('nan')

print(f"CKA (classical–quantum): {cka_val:.4f}")
print(f"Neighborhood Jaccard k=10: {jac_summary[10]['mean']:.4f}")
print(f"Top-5/10/20 anomaly overlap: {top5_ov:.4f}/{top10_ov:.4f}/{top20_ov:.4f}")
print(f"Quantum-only candidates: {n_quantum_only}")
print(f"QI rank stability (Spearman): {rank_stability:.4f}")
print(f"Permutation z-score: {z_score:.2f}")

## Answer 9 Research Questions

In [ ]:
def yn(condition): return "YES" if condition else "NO"

q1 = cka_val < 0.85
q2 = jac_summary[10]["mean"] < 0.5
q3 = top10_ov < 0.7
q4 = n_quantum_only >= 5
q5 = rank_stability >= 0.7
q6 = cross_df.shape[0] >= 2
q7 = depth_df['cka'].std() < 0.1
q8 = (depth_df['cka'].std() < 0.1) and q6
q9_row = perm_df[perm_df['seed'].astype(str)=='z_score']
q9 = float(q9_row['permutation_cka'].values[0]) > 1.0 if len(q9_row)>0 else False

answers = [
    f"Q1. Quantum/classical similarity structures substantially different? {yn(q1)} (CKA={cka_val:.4f})",
    f"Q2. Neighborhoods differ substantially? {yn(q2)} (mean Jaccard k=10: {jac_summary[10]['mean']:.4f})",
    f"Q3. Anomaly rankings differ? {yn(q3)} (top-10 overlap: {top10_ov:.4f})",
    f"Q4. Quantum-specific candidates exist? {yn(q4)} (count: {n_quantum_only})",
    f"Q5. Q-Interestingness ranking stable? {yn(q5)} (mean Spearman: {rank_stability:.4f})",
    f"Q6. Behavior transfers to 2nd dataset? {yn(q6)} (cross-dataset rows: {cross_df.shape[0]})",
    f"Q7. Feature-map depth affects result? {yn(not q7)} (depth CKA std: {depth_df['cka'].std():.4f})",
    f"Q8. Feature-map choice affects result? see feature_map_comparison.csv",
    f"Q9. Observed structure > null? {yn(q9)} (z-score vs permutation: {z_score:.2f})",
]
for a in answers:
    print(a)

## Save FINAL_RESEARCH_RESULTS.csv

In [ ]:
final_results = [{
    'dataset': 'Red Wine Quality (UCI)',
    'sample_size': N,
    'qubits': 4,
    'feature_map': 'ZZFeatureMap',
    'reps': 2,
    'kernel_alignment': round(cka_val, 6),
    'mean_neighborhood_jaccard': round(jac_summary[10]['mean'], 6),
    'top5_anomaly_overlap': round(top5_ov, 4),
    'top10_anomaly_overlap': round(top10_ov, 4),
    'top20_anomaly_overlap': round(top20_ov, 4),
    'number_quantum_only_candidates': n_quantum_only,
    'qi_rank_stability_spearman': round(rank_stability, 4),
    'permutation_z_score': round(z_score, 4),
}]
pd.DataFrame(final_results).to_csv('../outputs/tables/FINAL_RESEARCH_RESULTS.csv', index=False)
print("Saved FINAL_RESEARCH_RESULTS.csv")

## Save FINAL_EXPERIMENT_LOG.csv

In [ ]:
import datetime
exp_log = []
# Feature map comparison entries
for _, row in fmap_df.iterrows():
    exp_log.append({'experiment':'feature_map_robustness','dataset':'Red Wine Quality',
                    'feature_map':row['feature_map'],'reps':row['reps'],
                    'sample_size':N,'metric':'cka','value':row['cka']})
# Depth robustness entries
for _, row in depth_df.iterrows():
    exp_log.append({'experiment':'depth_robustness','dataset':'Red Wine Quality',
                    'feature_map':'ZZFeatureMap','reps':int(row['reps']),
                    'sample_size':N,'metric':'cka','value':row['cka']})
# Scalability entries
for _, row in scale_df.iterrows():
    exp_log.append({'experiment':'scalability','dataset':'Red Wine Quality',
                    'feature_map':'ZZFeatureMap','reps':2,
                    'sample_size':int(row['n']),'metric':'runtime_s','value':row['runtime_s']})
# Cross dataset
for _, row in cross_df.iterrows():
    exp_log.append({'experiment':'cross_dataset','dataset':row['dataset'],
                    'feature_map':row['feature_map'],'reps':row['reps'],
                    'sample_size':int(row['sample_size']),'metric':'cka','value':row['kernel_alignment']})
pd.DataFrame(exp_log).to_csv('../outputs/tables/FINAL_EXPERIMENT_LOG.csv', index=False)
print(f"Saved FINAL_EXPERIMENT_LOG.csv ({len(exp_log)} rows)")

## Generate RESEARCH_REPORT.md

In [ ]:
def interpret_conclusion(cka, jac, top10, n_qonly, stability):
    score = 0
    if cka < 0.7: score += 2
    elif cka < 0.85: score += 1
    if jac < 0.3: score += 2
    elif jac < 0.5: score += 1
    if top10 < 0.5: score += 2
    elif top10 < 0.7: score += 1
    if n_qonly >= 10: score += 2
    elif n_qonly >= 5: score += 1
    if stability >= 0.9: score += 1
    if score >= 7: return "A", "Strong evidence of complementary quantum exploratory structure"
    elif score >= 5: return "B", "Evidence of measurable but limited complementary structure"
    elif score >= 3: return "C", "Weak evidence / largely similar classical and quantum structure"
    else:          return "D", "Framework requires further refinement"

conclusion_cat, conclusion_text = interpret_conclusion(
    cka_val, jac_summary[10]["mean"], top10_ov, n_quantum_only, rank_stability)

report = f"""# Q-Interestingness: Quantum Exploratory Data Analysis Research Report

## 1. Abstract
This report presents the results of the Q-Interestingness framework applied to the UCI Red Wine
Quality dataset (N=1599, sampled to N=150 for quantum computation). The framework examines whether
quantum feature spaces (via ZZFeatureMap + FidelityQuantumKernel) produce complementary structural
information relative to classical exploratory data analysis.

Key finding: Classical–Quantum Kernel Alignment (CKA) = {cka_val:.4f}, Neighborhood Jaccard (k=10)
mean = {jac_summary[10]['mean']:.4f}, Top-10 anomaly overlap = {top10_ov:.4f}, Quantum-specific
exploratory candidates = {n_quantum_only}, Q-Interestingness rank stability = {rank_stability:.4f}.

**Conclusion category: {conclusion_cat} — {conclusion_text}**

## 2. Research Question
Does moving data from a classical feature space into a quantum feature space change what exploratory
analysis considers structurally unusual, and can this difference be quantified through a
Q-Interestingness framework?

## 3. Hypotheses
- H1: Quantum and classical similarity structures differ. [CKA={cka_val:.4f}]
- H2: Quantum and classical local neighborhoods differ. [Jaccard k=10 mean={jac_summary[10]['mean']:.4f}]
- H3: Quantum and classical anomaly rankings differ. [Top-10 overlap={top10_ov:.4f}]
- H4: Quantum-specific exploratory candidates exist. [Count={n_quantum_only}]
- H5: Q-Interestingness is stable under weight changes. [Spearman={rank_stability:.4f}]
- H6: Behavior transfers across datasets. [Tested on sklearn Wine dataset]

## 4. Dataset
- Dataset 1: UCI Red Wine Quality, 1599 observations, 11 features + quality target
- Dataset 2: sklearn Wine, 178 observations, 13 features, 3 classes (not used for kernel)
- Quantum features (4): {', '.join(WINE_QUANTUM_FEATURES)}

## 5. Preprocessing
- Features scaled to [0, π] for quantum encoding (MinMaxScaler)
- Features standardized (StandardScaler) for classical baseline
- No observations removed; duplicates documented (not silently dropped)
- Random seed: 42 throughout

## 6. Classical Methodology
- Standardization: StandardScaler
- Anomaly detection: IsolationForest (contamination=0.1)
- LOF for supplementary anomaly scoring
- Kernel: RBF kernel (gamma=1/n_features)
- Dimensionality reduction: PCA, UMAP

## 7. Quantum Methodology
- Feature map: ZZFeatureMap (feature_dimension=4, reps=2, entanglement=linear)
- Fidelity: ComputeUncompute with StatevectorSampler
- Kernel: FidelityQuantumKernel — K_ij = |<ψ(x_i)|ψ(x_j)>|²
- Sample size: N=150 (seed=42)

## 8. Q-Interestingness Definition
QI(x) = 0.40 * QAS(x) + 0.20 * CAS(x) + 0.20 * N(x) + 0.20 * B(x)
Where:
- QAS = 1 - mean quantum similarity (quantum anomaly score)
- CAS = classical IsolationForest anomaly score
- N = quantum novelty (1 - mean similarity to top-5 quantum neighbors)
- B = quantum boundary/heterogeneity (variance of kernel row)

## 9. Experimental Setup
- Quantum kernel: N=150 (seed=42, deterministic)
- Feature map robustness: ZZFeatureMap vs ZFeatureMap, reps=2
- Depth robustness: reps in [1, 2, 3]
- Permutation control: 10 independently permuted datasets (seeds 0–9)
- Scalability: N in [50, 100, 150, 200]
- Second dataset: sklearn Wine (N=150 subsample)

## 10. Results

| Metric | Value |
|--------|-------|
| Classical–Quantum CKA | {cka_val:.6f} |
| Neighborhood Jaccard k=5 | {jac_summary[5]['mean']:.4f} ± {jac_summary[5]['std']:.4f} |
| Neighborhood Jaccard k=10 | {jac_summary[10]['mean']:.4f} ± {jac_summary[10]['std']:.4f} |
| Neighborhood Jaccard k=15 | {jac_summary[15]['mean']:.4f} ± {jac_summary[15]['std']:.4f} |
| Top-5 anomaly overlap | {top5_ov:.4f} |
| Top-10 anomaly overlap | {top10_ov:.4f} |
| Top-20 anomaly overlap | {top20_ov:.4f} |
| Quantum-specific candidates | {n_quantum_only} |
| QI rank stability (Spearman) | {rank_stability:.4f} |
| Permutation z-score | {z_score:.4f} |

## 11. Robustness
Feature map comparison and depth robustness detailed in outputs/tables/feature_map_comparison.csv
and outputs/tables/depth_robustness.csv.

## 12. Permutation Control
Permutation mean CKA = {perm_mean:.4f}, std = {perm_std:.4f}.
Observed CKA = {cka_val:.4f}.
Standardized difference z = {z_score:.4f} (NOT a formal p-value).

## 13. Limitations
- StatevectorSampler simulates full quantum state classically; no actual quantum hardware.
- N=150 computational limit; full 1599-observation kernel not computed.
- ZZFeatureMap entanglement structure is fixed (linear); other structures untested.
- Q-Interestingness weights are heuristically chosen; no ground-truth validation.
- Quantum boundary (B) metric is a prototype heterogeneity measure, not theoretically proven.

## 14. Discussion
The Q-Interestingness framework provides a structured approach to examining complementary
exploratory structure between classical and quantum feature spaces. Results should be interpreted
with caution given the simulation-based quantum computation and the heuristic nature of the
composite score weights.

## 15. Conclusion
**Category: {conclusion_cat}**
{conclusion_text}.

This conclusion is based on the measured values above and should not be interpreted as claiming
quantum advantage. The CKA score of {cka_val:.4f} and neighborhood Jaccard of {jac_summary[10]['mean']:.4f}
quantify the degree of structural alignment between the two feature spaces.
"""

with open('../outputs/RESEARCH_REPORT.md', 'w') as f:
    f.write(report)
print("Saved RESEARCH_REPORT.md")
print("\n" + "="*60)
print("NOTEBOOK 06 COMPLETE — ALL OUTPUTS GENERATED")
print("="*60)